## Configure

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from os.path import join
from pathlib import Path
import json
import yaml
from yaml.loader import SafeLoader
import pickle
import pandas as pd
import geopandas as gpd
import numpy as np
from scipy import stats
from sklearn.metrics import mean_squared_error

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import matplotlib.gridspec as gridspec
from matplotlib.ticker import FuncFormatter
import matplotlib.ticker as mtick
import seaborn as sns

## Load data for analysis and plotting

In [ ]:
# Experiments and benchmarks

with open(join(FO, 'main_exp.pkl'), 'rb') as f:
    all_results = pickle.load(f)

with open(join(FO, 'benchmarks.pkl'), 'rb') as f:
    all_benchmarks = pickle.load(f)

In [ ]:
# Spatial data
clip_geo = gpd.read_file(CLIP_SHP_FILEP)
tract_ref = gpd.read_file(join(REF_DIR_I, FIPS, 'tract.gpkg'))[['GEOID', 'geometry']]

# Inventories
nsi_clip_out = gpd.read_file(join(EXP_DIR_I, FIPS, 'nsi_res.gpkg'))
phil_inv_out = gpd.read_file(join(EXP_DIR_I, FIPS, 'phil_res.gpkg'))

# Data to link inventories to tracts
phil_refs = pd.read_parquet(join(EXP_DIR_I, FIPS, 'phil_ref.pqt'))
nsi_refs = pd.read_parquet(join(EXP_DIR_I, FIPS, 'nsi_ref.pqt'))

## Postprocessing results for figures

There are a few datasets that will be helpful for analyzing and visualizing the results. 

First, we want to identify consistent records between the structure inventories and categorize records in terms of their match types across inventories. This is the basis for most of our main results. For each match type, we want to know about damage discrepancies, rank discrepancies, etc., so there is a lot of processing here. 

Second, we want to have results across all experiments summarized at the census tract level. A main inquiry of research is whether damage discrepancies at the property level cancel out at the scales most typical for using prospective damage estimates in support of decisions. 

Finally, we want to calculate how well the various experiments compare to the "best" representation of damages using the Philly inventory. 

In [ ]:
# Can only choose depth grids that we generated ensembles for
dg_id = '009'
d_min = 0
dam_col = 'naccs_loss_' + dg_id

### Get match types

First we calculate match types across inventories

In [ ]:
# Reference of nsi records linked to philly footprints
lnk_nsi_loc = gpd.sjoin(nsi_clip_out,
                        phil_inv_out,
                        predicate='within',
                        how='inner')

# Use the spatial links to add bfid to the nsi data
fd_bfid_lnk = dict(zip(lnk_nsi_loc['fd_id'], lnk_nsi_loc['bfid']))
nsi_match = nsi_inv_ens.reset_index()
nsi_match['bfid'] = nsi_match['fd_id'].map(fd_bfid_lnk)
# We will link up all the records across inventories
# including those w/o a match in the other in order to define
# different match categories
nsi_match.loc[nsi_match['bfid'].isnull(), 'bfid'] = nsi_match.loc[nsi_match['bfid'].isnull()]['fd_id']
match_df = nsi_match.merge(phil_inv_ens.reset_index(),
                           on='bfid',
                           suffixes=['_nsi', '_phil'],
                           how='outer')

matches = match_df.groupby(['bfid', 'num_story_phil', 'found_type_phil',
                            'occtype_phil', 'occtype_nsi',
                            'num_story_nsi', 'found_type_nsi']).size().reset_index()

matches['story_match'] = 0
matches.loc[matches['num_story_phil'] == matches['num_story_nsi'],
            'story_match'] = 1

matches['found_type_match'] = 0
matches.loc[matches['found_type_phil'] == matches['found_type_nsi'],
            'found_type_match'] = 1

matches['occ_match'] = 0
matches.loc[matches['occtype_phil'] == matches['occtype_nsi'],
            'occ_match'] = 1

matches.loc[(matches['story_match'] == 0) & (matches['found_type_match'] == 0),
            'Structure Matches'] = 'Location Only'

matches.loc[((matches['story_match'] == 1) &
             (matches['found_type_match'] == 1) &
             (matches['occ_match'] == 1)),
            'Structure Matches'] = 'All'

matches.loc[((matches['story_match'] == 1) &
             (matches['found_type_match'] == 1) &
             (matches['occ_match'] == 0)),
            'Structure Matches'] = 'Basement & Stories'

matches.loc[((matches['story_match'] == 1) &
             (matches['found_type_match'] == 0) &
             (matches['occ_match'] == 0)),
            'Structure Matches'] = 'Stories'

matches.loc[((matches['story_match'] == 0) &
             (matches['found_type_match'] == 1) &
             (matches['occ_match'] == 0)),
            'Structure Matches'] = 'Basement'

matches.loc[((matches['story_match'] == 0) &
             (matches['found_type_match'] == 0) &
             (matches['occ_match'] == 1)),
            'Structure Matches'] = 'Occupancy'

matches.loc[((matches['story_match'] == 1) &
             (matches['found_type_match'] == 0) &
             (matches['occ_match'] == 1)),
            'Structure Matches'] = 'Stories & Occupancy'

matches.loc[((matches['story_match'] == 0) &
             (matches['found_type_match'] == 1) &
             (matches['occ_match'] == 1)),
            'Structure Matches'] = 'Basement & Occuapncy'

match_agg_l = ['Stories', 'Basement', 'Occupancy', 'Stories & Occupancy',
               'Basement & Stories', 'Basement & Occuapncy']
matches.loc[matches['Structure Matches'].isin(match_agg_l),
            'Structure Matches'] = 'Location & Subset'

match_dict = dict(zip(matches['bfid'].astype(int), matches['Structure Matches']))

phil_only = match_df[(~match_df['bfid'].isin(matches['bfid'])) & (match_df['fd_id'].isnull())]
phil_only['Structure Matches'] = 'Unmatched Philly'
nsi_only = match_df[(~match_df['bfid'].isin(matches['bfid'])) & (match_df['fd_id'].notnull())]
nsi_only['Structure Matches'] = 'NSI Not In Philly'

phil_only_dict = dict(zip(phil_only['bfid'].astype(int), phil_only['Structure Matches']))
nsi_only_dict = dict(zip(nsi_only['bfid'].astype(int), nsi_only['Structure Matches']))

match_dict |= phil_only_dict
match_dict |= nsi_only_dict

Then we calculate damage discrepancies across inventories. In our case study, we are using the ensemble to estimate "best guess" damage estimates with the Philly inventory, so we are taking the discrepancy relative to the *mean*. 

In [ ]:
nsi_phil_id_dict = dict(zip(lnk_nsi_loc['fd_id'], lnk_nsi_loc['bfid']))
main_nsi = benchmarks['no_adj'].join(nsi_inv_ens).reset_index()

main_phil = all_results['original']['phil:no_adj']
# main_phil = main_phil.groupby('bfid')[dam_col].mean().reset_index()
main_phil = main_phil.merge(phil_inv_ens.reset_index(), on='bfid')

main_nsi['bfid'] = main_nsi['fd_id'].map(nsi_phil_id_dict)

# Calculate relative damage for each and include that in the mean step after merge
main_nsi['rel_loss'] = main_nsi[dam_col]/main_nsi['val_struct']
main_phil['rel_loss'] = main_phil[dam_col]/main_phil['val_s']

# Merge depths in
main_nsi[dg_id] = main_nsi['fd_id'].map(nsi_depths_df[dg_id])*3.28084
main_phil[dg_id] = main_phil['bfid'].map(phil_depths_df[dg_id])*3.28084

merge_cols = ['bfid', dam_col, dg_id, 'rel_loss', 'num_story', 'found_type', 'occtype']

# Do a quick update of occtype to RES1 if basement property
main_phil.loc[main_phil['found_type'] == 'B', 'occtype'] = 'RES1'

phil_mean = main_phil.groupby('bfid').agg({dam_col: 'mean',
                                           dg_id: 'first',
                                           'rel_loss': 'mean',
                                           'occtype': 'first',
                                           'num_story': 'first',
                                           'found_type': 'first'}).reset_index()

test = main_nsi.merge(phil_mean[merge_cols],
                      suffixes=['_nsi', '_phil'],
                      on='bfid',
                      how='outer')

# If bfid is null or Philly damage is 0, use fd_id in its place so we have a unique id
# We also look at Philly damage being 0 because if that's the case there's no depth
# for the property at that bfid and we want to use the fd_id instead for merging
# depths in later
fd_id_mask = test['bfid'].isnull() # | test[dam_col+'_phil'].isnull()
test.loc[fd_id_mask, 'bfid'] = test.loc[fd_id_mask, 'fd_id']

# NSI buildings can be stacked on top of each other so we'll aggregate these
# and treat them like one structure
test_gb = test.groupby(['bfid']).agg({dam_col + '_nsi': 'sum',
                                      dam_col + '_phil': 'first',
                                      dg_id + '_nsi': 'first',
                                      dg_id + '_phil': 'first',
                                      'rel_loss_nsi': 'first',
                                      'rel_loss_phil': 'first'}).reset_index().fillna(0)

test_gb['bfid'] = test_gb['bfid'].astype(int)

test_gb['nsi_rank'] = test_gb[dam_col+'_nsi'].rank(ascending=False, method='min')
test_gb['phil_rank'] = test_gb[dam_col+'_phil'].rank(ascending=False, method='min')

# Dam and rank diff
test_gb['diff'] = test_gb[dam_col+'_nsi'] - test_gb[dam_col+'_phil']
test_gb['diff_rel'] = test_gb['rel_loss_nsi'] - test_gb['rel_loss_phil']
test_gb['rank_diff'] = test_gb['nsi_rank'] - test_gb['phil_rank']
test_gb['diff_m'] = test_gb['diff']/1e6

# For visualization purposes, create an artificial high fillna value
test_gb.loc[test_gb['nsi_rank'].isnull(), 'nsi_rank'] = len(test_gb) 
test_gb.loc[test_gb['phil_rank'].isnull(), 'phil_rank'] = len(test_gb)


test_gb['Matches'] = test_gb['bfid'].map(match_dict)# .fillna('Philly Only (No Match)')

# Get depth back in 
test_gb['depth_ft'] = (test_gb['bfid'].map(phil_depths_df[dg_id])*3.28084)
test_gb.loc[test_gb['depth_ft'].isnull(),
            'depth_ft'] = test_gb['bfid'].map(nsi_depths_df[dg_id])*3.28084

# Id for "true" damage
test_gb['phil_dam'] = 0
test_gb.loc[test_gb['bfid'].isin(phil_mean['bfid']), 'phil_dam'] = 1

# Assign depths based on damage source
test_gb.loc[test_gb['phil_dam'] == 1, 'depth_ft'] = test_gb.loc[test_gb['phil_dam'] == 1, dg_id+'_phil']
test_gb.loc[test_gb['phil_dam'] == 0, 'depth_ft'] = test_gb.loc[test_gb['phil_dam'] == 0, dg_id+'_nsi']

# Update match column where no Philly damages
# test_gb.loc[test_gb['phil_dam'] == 0, 'Matches'] = 'NSI Only (No Match)'

# We only want to keep columns where there is damage in either record 
test_gb = test_gb.loc[(test_gb['phil_dam'] == 1) | (test_gb[dam_col+'_nsi'] > 0)]

# Get tract_id back in
test_gb['tract_id'] = test_gb['bfid'].map(phil_refs.set_index('bfid')['tract_id'])
test_gb.loc[test_gb['tract_id'].isnull(),
            'tract_id'] = test_gb['bfid'].map(nsi_refs.set_index('fd_id')['tract_id'])

# For cumulative discrepancies we are doing it using the mean from the ensemble
# as our best guess damage for the property

test_gb['depth_plot'] = test_gb['depth_ft'].round(1)
test_gb['cm_diff'] = test_gb.sort_values('depth_plot').groupby('Matches')['diff'].transform('cumsum')/1e6

test_gb['cm_diff_agg'] = test_gb.sort_values('depth_plot')['diff'].transform('cumsum')/1e6

### Census tract results 

In [ ]:
def create_comparison_dataframe(results_dict,
                                benchmark_dict,
                                phil_refs,
                                nsi_refs, 
                                phil_inventory,
                                nsi_inventory, 
                                dam_col,
                                ref_id,
                                result_keys=['phil:no_adj'],
                                benchmark_keys=['no_adj']):
    """
    Create a dataframe that links building IDs to different reference IDs
    for both Philadelphia and NSI data and aggregates damage and value
    to the level of the reference ID.
    
    Parameters:
    -----------
    results_dict : dict
        Dictionary containing ensemble results
    benchmark_dict : dict
        Dictionary containing benchmark results
    phil_refs : DataFrame
        DataFrame linking Philadelphia building IDs to reference IDs
    nsi_refs : DataFrame
        DataFrame linking NSI building IDs to reference IDs
    phil_inventory : DataFrame
        Philadelphia inventory data
    nsi_inventory : DataFrame
        NSI inventory data
    dam_col : str
        Column name for damage values
    ref_id: str
        Name of the spatial reference (e.g., "tract_id"). Must be in
        the phil_refs and nsi_refs dataframes
    result_keys : list, default=['phil:no_adj']
        Key to access specific results in results_dict
    benchmark_keys : list, default=['no_adj']
        Key to access specific benchmarks in benchmark_dict
        
    Returns:
    --------
    DataFrame
        Comparison dataframe with damage and property values for both datasets aggregated to
        the level of ref_id
    """
    # Process each result key
    result_dfs = {}
    for key in result_keys:
        # Determine which reference dataframe to use based on key prefix
        if key.startswith('phil'):
            refs = phil_refs
            id_col = 'bfid'
            inventory = phil_inventory
        else:
            refs = nsi_refs
            id_col = 'fd_id'
            inventory = nsi_inventory

        # Process ensemble results
        temp = results_dict[key]
        if ref_id not in temp.columns:
            temp = temp.merge(refs, on=id_col)

        temp_gb = temp.groupby(['sow_ind', ref_id]).agg({dam_col: 'sum'}).reset_index()
        loss_by_ref = temp_gb.groupby(ref_id)[dam_col].mean()
        
        # Calculate reference-level property values
        if ref_id not in inventory.columns:
            inventory = inventory.merge(refs[[id_col, ref_id]], on=id_col)
        
        ref_vals = inventory.groupby(ref_id).agg({'val_struct': ['median', 'sum', 'size']})
        ref_vals = ref_vals.reset_index()
        ref_vals.columns = [ref_id, 'median_val', 'total_val', 'n_prop']
        
        # Create result dataframe
        result_df = pd.DataFrame({
            dam_col: loss_by_ref,
            'median_val': ref_vals.set_index(ref_id)['median_val'],
            'total_val': ref_vals.set_index(ref_id)['total_val'],
            'n_prop': ref_vals.set_index(ref_id)['n_prop']
        })
        
        result_df = result_df[result_df[dam_col].notnull()]
        result_dfs[key] = result_df
   
    # Process each benchmark key
    benchmark_dfs = {}
    for key in benchmark_keys:
        refs = nsi_refs
        id_col = 'fd_id'
        inventory = nsi_inventory

    # Process benchmark results
        temp = benchmark_dict[key].reset_index()
        
        if ref_id not in inventory.columns:
            inventory = inventory.merge(refs[[id_col, ref_id]], on=id_col)
        
        if id_col in temp.columns:
            temp_w_refs = temp.merge(inventory, on=id_col)
            temp_gb = temp_w_refs.groupby([ref_id])[[dam_col]].sum().reset_index()
        else:
            # If benchmark already has ref_id, just group by it
            temp_gb = temp.groupby([ref_id])[[dam_col]].sum().reset_index()
        
        # Calculate reference-level property values
        ref_vals = inventory.groupby(ref_id).agg({'val_struct': ['median', 'sum', 'size']})
        ref_vals = ref_vals.reset_index()
        ref_vals.columns = [ref_id, 'median_val', 'total_val', 'n_prop']
        
        # Merge with property values
        temp_gb = temp_gb.merge(ref_vals, on=ref_id)
        benchmark_dfs[key] = temp_gb.set_index(ref_id)

    # Combine all dataframes
    all_dfs = []
    
    # Process result dataframes
    for key, df in result_dfs.items():
        df_reset = df.reset_index()
        df_reset.columns = [ref_id] + [f"{col}_{key.split(':')[0]}" for col in df.columns]
        all_dfs.append(df_reset)
    
    # Process benchmark dataframes
    for key, df in benchmark_dfs.items():
        df_reset = df.reset_index()
        df_reset.columns = [ref_id] + [f"{col}_nsi" for col in df.columns]
        all_dfs.append(df_reset)

    # Merge all dataframes
    if all_dfs:
        result = all_dfs[0]
        for df in all_dfs[1:]:
            result = result.merge(df, on=ref_id, how='outer')
        
        return result.fillna(0)
    else:
        return pd.DataFrame()


In [ ]:
comp = create_comparison_dataframe(
    all_results['original'], 
    all_benchmarks['original'],
    phil_refs,
    nsi_refs,
    phil_inv_ens,
    nsi_inv_ens,
    dam_col=dam_col,
    ref_id='tract_id',
    result_keys=['phil:no_adj', 'nsi_ddfs:no_adj', 'nsi_unsafe:no_adj',
                 'nsi_phil:no_adj', 'phil_unsafe:no_adj', 'phil_nsi:no_adj'],
    benchmark_keys=['no_adj']
)

comp_geo = tract_ref.merge(comp, left_on='GEOID', right_on='tract_id')

### Skill metrics for each experiment compared to baseline

In [ ]:
def calculate_metrics(comp_df, exps, dam_col, baseline='phil'):
    """Calculate various skill metrics for each experiment compared to baseline."""
    metrics = {}
    
    # Baseline values
    baseline_loss = comp_df[f"{dam_col}_{baseline}"]/1e6
    baseline_val = comp_df[f"median_val_{baseline}"]
    baseline_rank = baseline_loss.rank(ascending=False)
    baseline_total = baseline_loss.sum()
    
    # Top 20% tracts in baseline
    top20_threshold = int(len(comp_df) * 0.1)
    top20_tracts = baseline_loss.nlargest(top20_threshold).index
    
    for exp in exps:
        exp_metrics = {}
        
        # Calculate losses and ranks
        exp_loss = comp_df[f"{dam_col}_{exp}"]/1e6
        exp_val = comp_df[f"median_val_{exp}"]
        exp_rank = exp_loss.rank(ascending=False)
        exp_total = exp_loss.sum()
        
        # Total discrepancy metrics
        exp_metrics['total_discrepancy_dollar'] = exp_total - baseline_total
        exp_metrics['total_discrepancy_pct'] = 100 * (exp_total - baseline_total) / baseline_total
        
        # RMSE metrics
        exp_metrics['rmse_tract'] = np.sqrt(mean_squared_error(baseline_loss, exp_loss))
        
        # Correlation between structure value and damages

        exp_metrics['corr_val_dam'] = (stats.pearsonr(exp_loss, exp_val)[0] -
                                       stats.pearsonr(baseline_loss, baseline_val)[0])
        

        # Rank correlation metrics
        exp_metrics['rank_correlation'] = stats.spearmanr(exp_rank, baseline_rank)[0]
        
        # Top percentage rank correlation
        exp_metrics['rank_correlation_top20'] = stats.spearmanr( 
            exp_rank[baseline_rank <= len(top20_tracts)],
            baseline_rank[baseline_rank <= len(top20_tracts)]
        )[0]
        
        # Type 1 and Type 2 errors for top 20%
        # Type 1: Baseline says it's top 20%, experiment says it's not
        type1_error = len(set(top20_tracts) - set(exp_loss.nlargest(top20_threshold).index))
        exp_metrics['type1_error_count'] = type1_error 
        exp_metrics['type1_pct'] = 100*type1_error/top20_threshold
        
        # Type 2: Experiment says it's top 20%, baseline says it's not
        type2_error = len(set(exp_loss.nlargest(top20_threshold).index) - set(top20_tracts))
        exp_metrics['type2_error_count'] = type2_error 
        exp_metrics['type2_pct'] = 100*type2_error/top20_threshold
        
        # Count of matched ranks
        exp_metrics['matched_top_rank_pct'] = (
             exp_rank[exp_rank <= len(top20_tracts)] -
             baseline_rank[baseline_rank <= len(top20_tracts)] == 0
            ).sum()*100/len(top20_tracts)


        # std error of ranks
        exp_metrics['std_rank'] = np.std(
            baseline_rank - exp_rank
        )

        metrics[exp] = exp_metrics
    
    return metrics

In [ ]:
# Calculate metrics
exps = ['nsi', 'nsi_ddfs', 'nsi_unsafe', 'nsi_phil', 'phil_nsi', 'phil_unsafe']
metrics = calculate_metrics(comp, exps, dam_col)

## Figures